# EVA-CLM: T4 pre-A100 notebook (dry-run prod + mid-scale A/B)

Бюджет: Colab T4 (~2ч). Ноутбук НЕ обучает прод (в T4 не влезет без gc и даёт мало шагов) —
он делает две вещи:
1. **dry-run прода** (5-10 мин): влезает ли прод-конфиг (D=2560/24L) в 16GB с gc=True+AMP,
   и какая скорость. Порог: s/step > 30 -> прод на T4 не тратить.
2. **mid-scale A/B** (~60-95 мин): bounded_residual=True vs False на D=512/8L, реальные
   данные — проверка переделки на большем, чем мини, масштабе.

Перед запуском: Runtime -> Change runtime type -> T4 GPU.


In [ ]:
# 0. Репозиторий: укажи свой путь/клон (как в основном ноутбуке)
import os, subprocess
REPO = '/content/eva_clm'   # <-- ЗАМЕНИ на свой путь к репо (или Drive-путь)
if not os.path.isdir(REPO):
    os.makedirs('/content', exist_ok=True)
    # раскомментируй и подставь URL, если клонируешь:
    # subprocess.run(['git', 'clone', '<REPO_URL>', REPO], check=True)
    raise SystemExit(f'Нет репо: {REPO} — поправь REPO/клон')
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
print('cwd:', os.getcwd())


In [ ]:
# 1. Данные: нужны стримы wb/token_stream_*_eos.bin (3 train + WAR hold-out)
import glob, os
wb = sorted(glob.glob(os.path.join(REPO, 'wb', 'token_stream_*_eos.bin')))
print('streams:', len(wb))
for p in wb[:6]:
    print(' ', os.path.basename(p), round(os.path.getsize(p)/1e6, 1), 'MB')
holdout = os.path.join(REPO, 'wb', 'token_stream_WAR_eos.bin')
assert os.path.exists(holdout), 'нет WAR hold-out: ' + holdout
print('holdout OK:', holdout)


In [ ]:
# 2. DRY-RUN прода (5-10 мин): peak VRAM, s/step, b_flow/b_drift
!python scripts/t4_bounded_ab.py --dry-run --data-dir {REPO} --out-dir {REPO}/logs/t4 --out /content/t4_dry.json
import json, os
p = '/content/t4_dry.json'
if os.path.exists(p):
    d = json.load(open(p))
    print('=== DRY-RUN ===')
    print(json.dumps(d, indent=1, ensure_ascii=False)[:3000])
    s = d.get('s_per_step') or d.get('sec_per_step')
    if s and float(s) > 30:
        print('>>> s/step > 30: прод на T4 НЕ тратить (шагов за 2ч слишком мало)')
else:
    print('нет /content/t4_dry.json — смотри консоль выше')


In [ ]:
# 3. MID-SCALE A/B: bounded_residual=True vs False (~60-95 мин)
!python scripts/t4_bounded_ab.py --ab --max-minutes 95 --data-dir {REPO} --out-dir {REPO}/logs/t4 --holdout {REPO}/wb/token_stream_WAR_eos.bin


In [ ]:
# 4. Результаты: сырые JSON/txt (пришли их мне для разбора)
import glob, os, json
for f in sorted(glob.glob('logs/t4/*')):
    print('=' * 20, f)
    if f.endswith('.json'):
        try:
            d = json.load(open(f))
            print(json.dumps(d, indent=1, ensure_ascii=False)[:4000])
        except Exception as e:
            print('json error:', e)
    elif f.endswith('.txt'):
        print(open(f, encoding='utf-8', errors='replace').read()[:4000])
